# Agent 1 — Module 2: Exact Notebook Migration

## Objective

This notebook migrates the current Module 2 implementation into Jupyter without changing the existing chunking logic.

The following project files are copied into executable notebook cells exactly as they currently exist:

1. `app/schemas/chunk.py`
2. `app/services/embedding_service.py`
3. `app/services/semantic_chunker.py`
4. `scripts/test_semantic_chunking.py`

No semantic boundary, chunk plan, overlap, continuation relationship, threshold, or expected chunk count is manually assigned.

The only experimental change is the embedding model configuration:

```text
Qwen/Qwen3-Embedding-0.6B
```

versus:

```text
sentence-transformers/all-MiniLM-L6-v2
```

Both models use the same existing `SemanticChunker` code and the same real cleaned transcript.

# 0. Environment Setup

Run this notebook from the `Agent_1` project root.

The setup installs the project requirements and ensures the project package can be imported.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [
        start,
        *start.parents,
        start / "Agent_1",
    ]:
        if (
            (candidate / "app").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate.resolve()

    raise RuntimeError(
        "Agent_1 project root was not found."
    )


PROJECT_ROOT = find_project_root(
    Path.cwd().resolve()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(PROJECT_ROOT / "requirements.txt"),
    ]
)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "numpy",
        "pandas",
        "sentence-transformers",
    ]
)

print(f"Python: {sys.executable}")
print(f"Project root: {PROJECT_ROOT}")

# 1. Exact Existing Chunk Schemas

The next code cell is copied directly from:

```text
app/schemas/chunk.py
```

No line of the schema logic has been rewritten.

In [ ]:
from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field


BoundaryReason = Literal[
    "semantic_shift",
    "transition_phrase",
    "semantic_shift+transition_phrase",
    "max_size",
    "end_of_transcript",
]

TransitionStrength = Literal[
    "strong",
    "soft",
]

SegmentPosition = Literal[
    "single",
    "start",
    "middle",
    "end",
]

ContinuationReason = Literal[
    "max_size_split",
]


class TranscriptChunk(BaseModel):
    """
    One meaningful section of a lesson transcript.

    `chunk_id` identifies the physical chunk.

    `segment_id` identifies the larger logical lesson segment. Multiple
    chunks can belong to the same segment when the chunker is forced to
    split a long, continuous discussion because of max_chunk_words.
    """

    chunk_id: int = Field(ge=1)

    # Actual text passed downstream.
    # For a max-size split, this may include a small overlap from
    # the previous chunk for context preservation.
    text: str = Field(min_length=1)

    word_count: int = Field(ge=1)
    sentence_count: int = Field(ge=1)

    # Sentence range represented by `text`.
    # This may overlap the previous chunk only after a forced max-size split.
    start_sentence: int = Field(ge=0)
    end_sentence: int = Field(ge=0)

    # Non-overlapping/core sentence range belonging to this chunk.
    core_start_sentence: int = Field(ge=0)
    core_end_sentence: int = Field(ge=0)

    # Why this chunk ended.
    boundary_reason: BoundaryReason

    # Similarity between semantic units around the ending boundary.
    boundary_similarity: float | None = Field(
        default=None,
        ge=-1.0,
        le=1.0,
    )

    # Whether a transition phrase supported the ending boundary.
    boundary_transition_strength: TransitionStrength | None = None

    # Context repeated from the previous chunk.
    # Non-zero only when the previous chunk ended because of max_size.
    overlap_word_count: int = Field(
        default=0,
        ge=0,
    )

    # -------------------------------------------------------------
    # Logical segment and continuation metadata
    # -------------------------------------------------------------

    # Human-readable logical segment identifier.
    segment_id: str = Field(
        default="segment_001",
        min_length=1,
    )

    # First physical chunk belonging to this logical segment.
    segment_root_chunk_id: int = Field(
        default=1,
        ge=1,
    )

    # Position of this chunk inside its logical segment.
    segment_chunk_index: int = Field(
        default=1,
        ge=1,
    )

    # Total number of physical chunks in the logical segment.
    segment_chunk_count: int = Field(
        default=1,
        ge=1,
    )

    segment_position: SegmentPosition = "single"

    # True when this chunk continues the same logical segment because
    # the previous physical chunk was forcibly split at max size.
    is_continuation: bool = False

    # Immediate previous physical chunk continued by this chunk.
    continuation_of_chunk_id: int | None = Field(
        default=None,
        ge=1,
    )

    continuation_reason: ContinuationReason | None = None


class ChunkingResult(BaseModel):
    """Complete output of Module 2."""

    chunks: list[TranscriptChunk]

    total_sentences: int = Field(ge=0)
    total_words: int = Field(ge=0)
    semantic_unit_count: int = Field(ge=0)

    # Number of logical lesson segments after grouping forced
    # max-size continuations.
    segment_count: int = Field(
        default=0,
        ge=0,
    )

    embedding_model: str

    # Actual semantic threshold calculated for this transcript.
    semantic_threshold: float

    # Effective configuration for reproducible testing.
    min_chunk_words: int = Field(ge=1)
    target_chunk_words: int = Field(ge=1)
    max_chunk_words: int = Field(ge=1)
    max_size_overlap_words: int = Field(ge=0)

# 2. Exact Existing Embedding Service

The next code cell is copied directly from:

```text
app/services/embedding_service.py
```

It already defines both models:

```python
DEFAULT_EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
CHUNKING_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
```

No embedding logic has been changed.

In [ ]:
from __future__ import annotations

import os
from functools import lru_cache
from typing import Sequence

import numpy as np
from sentence_transformers import SentenceTransformer


# Main/high-accuracy embedding model selected for curriculum retrieval/mapping.
DEFAULT_EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"

# Lightweight model used specifically for Module 2 semantic chunking.
CHUNKING_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


@lru_cache(maxsize=4)
def get_embedding_model(
    model_name: str = DEFAULT_EMBEDDING_MODEL,
) -> SentenceTransformer:
    """
    Load and cache embedding models.

    Multiple models may be cached in the same process:
    - MiniLM for fast transcript chunking
    - Qwen3-Embedding-0.6B for curriculum retrieval/mapping
    """

    device = os.getenv("EMBEDDING_DEVICE")

    kwargs: dict[str, str] = {}

    if device:
        kwargs["device"] = device

    return SentenceTransformer(
        model_name,
        **kwargs,
    )


def embed_texts(
    texts: Sequence[str],
    model_name: str = DEFAULT_EMBEDDING_MODEL,
    batch_size: int = 32,
) -> np.ndarray:
    """
    Generate normalized embeddings for multiple pieces of text.

    Normalized embeddings allow cosine similarity to be calculated
    efficiently using a dot product.
    """

    cleaned_texts = [
        str(text).strip()
        for text in texts
        if str(text).strip()
    ]

    if not cleaned_texts:
        return np.empty(
            (0, 0),
            dtype=np.float32,
        )

    model = get_embedding_model(
        model_name
    )

    embeddings = model.encode(
        cleaned_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    return np.asarray(
        embeddings,
        dtype=np.float32,
    )

# 3. Exact Existing Semantic Chunker

The next code cell is copied directly from:

```text
app/services/semantic_chunker.py
```

This is the same implementation used by the existing Agent 1 pipeline.

It performs:

- sentence splitting;
- semantic-unit construction;
- embedding generation;
- neighbour similarity calculation;
- adaptive threshold calculation;
- transition detection;
- guarded boundary planning;
- maximum-size splitting;
- conditional overlap;
- logical segment assignment.

No internal function has been rewritten for the notebook.

In [ ]:
from __future__ import annotations

import re
from collections import defaultdict
from dataclasses import dataclass

import numpy as np

from app.schemas.chunk import (
    ChunkingResult,
    TranscriptChunk,
)
from app.services.embedding_service import (
    CHUNKING_EMBEDDING_MODEL,
    embed_texts,
)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class SemanticChunkingConfig:
    """
    Configuration for guarded semantic chunking.

    Design goals:
    - avoid over-fragmenting teacher/student discussion
    - respect explicit lesson transitions
    - prevent excessively large chunks
    - use overlap only when size forces a split
    - add deterministic logical segment metadata
    - use MiniLM-specific adaptive similarity thresholds
    """

    min_chunk_words: int = 150
    target_chunk_words: int = 325
    max_chunk_words: int = 550

    strong_transition_min_words: int = 80
    semantic_unit_words: int = 60

    boundary_percentile: float = 15.0
    threshold_floor: float = 0.10
    threshold_ceiling: float = 0.45

    soft_transition_margin: float = 0.10
    soft_transition_similarity_ceiling: float = 0.35

    size_penalty_weight: float = 0.12
    strong_transition_bonus: float = 0.10
    soft_transition_bonus: float = 0.04

    max_size_overlap_words: int = 45
    max_size_overlap_sentences: int = 2

    embedding_model: str = CHUNKING_EMBEDDING_MODEL

    def __post_init__(self) -> None:
        if not (
            0
            < self.strong_transition_min_words
            <= self.min_chunk_words
            <= self.target_chunk_words
            <= self.max_chunk_words
        ):
            raise ValueError(
                "Chunk sizes must satisfy: "
                "0 < strong_transition_min <= min <= target <= max."
            )

        if self.semantic_unit_words <= 0:
            raise ValueError(
                "semantic_unit_words must be positive."
            )

        if not 0 <= self.boundary_percentile <= 100:
            raise ValueError(
                "boundary_percentile must be between 0 and 100."
            )

        if not -1.0 <= self.threshold_floor <= 1.0:
            raise ValueError(
                "threshold_floor must be between -1 and 1."
            )

        if not -1.0 <= self.threshold_ceiling <= 1.0:
            raise ValueError(
                "threshold_ceiling must be between -1 and 1."
            )

        if self.threshold_floor > self.threshold_ceiling:
            raise ValueError(
                "threshold_floor cannot be greater than threshold_ceiling."
            )

        if self.soft_transition_margin < 0:
            raise ValueError(
                "soft_transition_margin cannot be negative."
            )

        if not -1.0 <= self.soft_transition_similarity_ceiling <= 1.0:
            raise ValueError(
                "soft_transition_similarity_ceiling must be between -1 and 1."
            )

        if self.max_size_overlap_words < 0:
            raise ValueError(
                "max_size_overlap_words cannot be negative."
            )

        if self.max_size_overlap_sentences < 0:
            raise ValueError(
                "max_size_overlap_sentences cannot be negative."
            )


# ---------------------------------------------------------------------
# Internal structures
# ---------------------------------------------------------------------


@dataclass(frozen=True)
class _SemanticUnit:
    text: str
    start_sentence: int
    end_sentence: int
    word_count: int

    # Transition at the START of this unit.
    transition_strength: str | None = None


@dataclass(frozen=True)
class _Boundary:
    """Candidate boundary occurring AFTER unit_index."""

    unit_index: int
    similarity: float
    reason: str
    transition_strength: str | None = None


@dataclass(frozen=True)
class _ChunkPlan:
    """
    Non-overlapping/core chunk boundaries.

    Overlap and logical segment metadata are added while materializing
    final TranscriptChunk objects.
    """

    start_unit: int
    end_unit: int
    reason: str
    similarity: float | None
    transition_strength: str | None = None


# ---------------------------------------------------------------------
# Semantic Chunker
# ---------------------------------------------------------------------


class SemanticChunker:
    """
    Module 2.

    Input:
        Final cleaned transcript from Module 1.

    Output:
        Meaningful physical chunks plus logical segment metadata.

    A logical segment may contain multiple physical chunks when a long,
    continuous discussion must be split only because of max_chunk_words.

    This module does NOT:
    - identify topics
    - map to the AQA syllabus
    - call an LLM
    - store transcript embeddings
    """

    STRONG_TRANSITION_PATTERNS = (
        r"\bnext (?:chapter|topic|concept|section|question)\b",
        r"\bthe next (?:chapter|topic|concept|section|question)\b",
        r"\bstart (?:the )?(?:next )?chapter\b",
        r"\bbegin (?:the )?(?:next )?chapter\b",
        r"\bchapter (?:number )?\d+\b",
        r"\bnew (?:chapter|topic|concept|section)\b",
        r"\bmove on to (?:the )?(?:next )?(?:chapter|topic|concept|section)\b",
        r"\blet'?s move on to (?:the )?next (?:thing|one|question)\b",
        r"\bmove on to (?:the )?next (?:thing|one|question)\b",
        r"\blet'?s look at (?:the )?next question\b",
        r"\blet'?s go to (?:the )?next question\b",
    )

    SOFT_TRANSITION_PATTERNS = (
        r"\blet'?s move on\b",
        r"\bmove on to\b",
        r"\bmoving on\b",
        r"\blet'?s talk about\b",
        r"\blet'?s discuss\b",
        r"\blet'?s look at\b",
        r"\bnext one\b",
        r"\bnext thing\b",
        r"\bnow (?:we can|we will|let'?s) move\b",
        r"\bnow (?:we can|we will|let'?s) (?:talk|discuss|look)\b",
    )

    def __init__(
        self,
        config: SemanticChunkingConfig | None = None,
    ) -> None:
        self.config = (
            config
            or SemanticChunkingConfig()
        )

    # -----------------------------------------------------------------
    # Public API
    # -----------------------------------------------------------------

    def chunk(
        self,
        cleaned_transcript: str,
    ) -> ChunkingResult:
        """Convert a cleaned transcript into guarded semantic chunks."""

        if not isinstance(
            cleaned_transcript,
            str,
        ):
            raise TypeError(
                "cleaned_transcript must be a string."
            )

        cleaned_transcript = cleaned_transcript.strip()

        if not cleaned_transcript:
            raise ValueError(
                "Cannot chunk an empty transcript."
            )

        sentences = self._split_sentences(
            cleaned_transcript
        )

        if not sentences:
            raise ValueError(
                "No usable sentences found in transcript."
            )

        units = self._build_semantic_units(
            sentences
        )

        if not units:
            raise ValueError(
                "No semantic units could be created."
            )

        if len(units) == 1:
            unit = units[0]

            chunk = TranscriptChunk(
                chunk_id=1,
                text=unit.text,
                word_count=unit.word_count,
                sentence_count=(
                    unit.end_sentence
                    - unit.start_sentence
                    + 1
                ),
                start_sentence=unit.start_sentence,
                end_sentence=unit.end_sentence,
                core_start_sentence=unit.start_sentence,
                core_end_sentence=unit.end_sentence,
                boundary_reason="end_of_transcript",
                boundary_similarity=None,
                boundary_transition_strength=None,
                overlap_word_count=0,
                segment_id="segment_001",
                segment_root_chunk_id=1,
                segment_chunk_index=1,
                segment_chunk_count=1,
                segment_position="single",
                is_continuation=False,
                continuation_of_chunk_id=None,
                continuation_reason=None,
            )

            return self._build_result(
                chunks=[chunk],
                sentences=sentences,
                units=units,
                cleaned_transcript=cleaned_transcript,
                semantic_threshold=self.config.threshold_ceiling,
            )

        unit_embeddings = embed_texts(
            [
                unit.text
                for unit in units
            ],
            model_name=self.config.embedding_model,
        )

        similarities = self._neighbour_similarities(
            unit_embeddings
        )

        semantic_threshold = self._calculate_threshold(
            similarities
        )

        boundaries = self._detect_boundaries(
            units=units,
            similarities=similarities,
            semantic_threshold=semantic_threshold,
        )

        plans = self._build_chunk_plans(
            units=units,
            similarities=similarities,
            boundaries=boundaries,
        )

        chunks = self._materialize_chunks(
            plans=plans,
            units=units,
            sentences=sentences,
        )

        return self._build_result(
            chunks=chunks,
            sentences=sentences,
            units=units,
            cleaned_transcript=cleaned_transcript,
            semantic_threshold=semantic_threshold,
        )

    def _build_result(
        self,
        chunks: list[TranscriptChunk],
        sentences: list[str],
        units: list[_SemanticUnit],
        cleaned_transcript: str,
        semantic_threshold: float,
    ) -> ChunkingResult:
        segment_count = len(
            {
                chunk.segment_id
                for chunk in chunks
            }
        )

        return ChunkingResult(
            chunks=chunks,
            total_sentences=len(sentences),
            total_words=self._word_count(
                cleaned_transcript
            ),
            semantic_unit_count=len(units),
            segment_count=segment_count,
            embedding_model=self.config.embedding_model,
            semantic_threshold=round(
                semantic_threshold,
                4,
            ),
            min_chunk_words=self.config.min_chunk_words,
            target_chunk_words=self.config.target_chunk_words,
            max_chunk_words=self.config.max_chunk_words,
            max_size_overlap_words=(
                self.config.max_size_overlap_words
            ),
        )

    # -----------------------------------------------------------------
    # Sentence preparation
    # -----------------------------------------------------------------

    @staticmethod
    def _split_sentences(
        text: str,
    ) -> list[str]:
        """
        Lightweight sentence segmentation.

        A period inside `array.length` is not split because there is no
        whitespace after that period.
        """

        text = text.replace(
            "\r",
            "\n",
        )

        text = re.sub(
            r"[ \t]+",
            " ",
            text,
        )

        text = re.sub(
            r"\s*\n+\s*",
            " ",
            text,
        )

        parts = re.split(
            r"(?<=[.!?])\s+",
            text.strip(),
        )

        return [
            part.strip()
            for part in parts
            if part.strip()
        ]

    # -----------------------------------------------------------------
    # Transition detection
    # -----------------------------------------------------------------

    def _transition_strength(
        self,
        sentence: str,
    ) -> str | None:
        head = sentence[:220].lower()

        if any(
            re.search(pattern, head)
            for pattern
            in self.STRONG_TRANSITION_PATTERNS
        ):
            return "strong"

        if any(
            re.search(pattern, head)
            for pattern
            in self.SOFT_TRANSITION_PATTERNS
        ):
            return "soft"

        return None

    # -----------------------------------------------------------------
    # Semantic units
    # -----------------------------------------------------------------

    def _build_semantic_units(
        self,
        sentences: list[str],
    ) -> list[_SemanticUnit]:
        units: list[_SemanticUnit] = []

        buffer: list[str] = []
        buffer_words = 0
        start_sentence = 0
        buffer_transition_strength: str | None = None

        def flush_buffer(
            end_sentence: int,
        ) -> None:
            nonlocal buffer
            nonlocal buffer_words
            nonlocal start_sentence
            nonlocal buffer_transition_strength

            if not buffer:
                return

            units.append(
                _SemanticUnit(
                    text=" ".join(buffer),
                    start_sentence=start_sentence,
                    end_sentence=end_sentence,
                    word_count=buffer_words,
                    transition_strength=(
                        buffer_transition_strength
                    ),
                )
            )

            buffer = []
            buffer_words = 0
            buffer_transition_strength = None

        for sentence_index, sentence in enumerate(
            sentences
        ):
            transition_strength = self._transition_strength(
                sentence
            )

            if transition_strength and buffer:
                flush_buffer(
                    sentence_index - 1
                )

            if not buffer:
                start_sentence = sentence_index
                buffer_transition_strength = (
                    transition_strength
                )

            buffer.append(sentence)
            buffer_words += self._word_count(
                sentence
            )

            if (
                buffer_words
                >= self.config.semantic_unit_words
            ):
                flush_buffer(
                    sentence_index
                )

        if buffer:
            flush_buffer(
                len(sentences) - 1
            )

        if len(units) >= 2:
            minimum_tail = max(
                15,
                self.config.semantic_unit_words
                // 2,
            )

            last = units[-1]

            if (
                last.word_count < minimum_tail
                and last.transition_strength is None
            ):
                previous = units[-2]

                units[-2] = _SemanticUnit(
                    text=(
                        previous.text
                        + " "
                        + last.text
                    ),
                    start_sentence=(
                        previous.start_sentence
                    ),
                    end_sentence=(
                        last.end_sentence
                    ),
                    word_count=(
                        previous.word_count
                        + last.word_count
                    ),
                    transition_strength=(
                        previous.transition_strength
                    ),
                )

                units.pop()

        return units

    # -----------------------------------------------------------------
    # Similarity
    # -----------------------------------------------------------------

    @staticmethod
    def _neighbour_similarities(
        embeddings: np.ndarray,
    ) -> np.ndarray:
        if len(embeddings) < 2:
            return np.array(
                [],
                dtype=np.float32,
            )

        similarities = np.sum(
            embeddings[:-1]
            * embeddings[1:],
            axis=1,
        )

        return similarities.astype(
            np.float32
        )

    def _calculate_threshold(
        self,
        similarities: np.ndarray,
    ) -> float:
        if len(similarities) == 0:
            return self.config.threshold_ceiling

        percentile_value = float(
            np.percentile(
                similarities,
                self.config.boundary_percentile,
            )
        )

        threshold = max(
            self.config.threshold_floor,
            min(
                self.config.threshold_ceiling,
                percentile_value,
            ),
        )

        return round(
            threshold,
            4,
        )

    def _soft_transition_threshold(
        self,
        semantic_threshold: float,
    ) -> float:
        return min(
            self.config.soft_transition_similarity_ceiling,
            semantic_threshold
            + self.config.soft_transition_margin,
        )

    # -----------------------------------------------------------------
    # Boundary detection
    # -----------------------------------------------------------------

    def _detect_boundaries(
        self,
        units: list[_SemanticUnit],
        similarities: np.ndarray,
        semantic_threshold: float,
    ) -> dict[int, _Boundary]:
        boundaries: dict[int, _Boundary] = {}

        soft_threshold = self._soft_transition_threshold(
            semantic_threshold
        )

        for index, similarity_value in enumerate(
            similarities
        ):
            similarity = float(
                similarity_value
            )

            semantic_shift = (
                similarity
                <= semantic_threshold
            )

            next_unit = units[index + 1]

            transition_strength = (
                next_unit.transition_strength
            )

            strong_transition = (
                transition_strength == "strong"
            )

            soft_transition = (
                transition_strength == "soft"
                and similarity <= soft_threshold
            )

            transition_boundary = (
                strong_transition
                or soft_transition
            )

            if not (
                semantic_shift
                or transition_boundary
            ):
                continue

            if (
                semantic_shift
                and transition_boundary
            ):
                reason = (
                    "semantic_shift+transition_phrase"
                )
            elif transition_boundary:
                reason = "transition_phrase"
            else:
                reason = "semantic_shift"

            boundaries[index] = _Boundary(
                unit_index=index,
                similarity=similarity,
                reason=reason,
                transition_strength=(
                    transition_strength
                    if transition_boundary
                    else None
                ),
            )

        return boundaries

    # -----------------------------------------------------------------
    # Chunk planning
    # -----------------------------------------------------------------

    def _build_chunk_plans(
        self,
        units: list[_SemanticUnit],
        similarities: np.ndarray,
        boundaries: dict[int, _Boundary],
    ) -> list[_ChunkPlan]:
        prefix_words = [0]

        for unit in units:
            prefix_words.append(
                prefix_words[-1]
                + unit.word_count
            )

        def words_between(
            start: int,
            end: int,
        ) -> int:
            return (
                prefix_words[end + 1]
                - prefix_words[start]
            )

        plans: list[_ChunkPlan] = []
        start = 0
        number_of_units = len(units)

        while start < number_of_units:
            remaining_words = words_between(
                start,
                number_of_units - 1,
            )

            candidates: list[
                tuple[int, _Boundary]
            ] = []

            for end_index in range(
                start,
                number_of_units - 1,
            ):
                current_words = words_between(
                    start,
                    end_index,
                )

                if (
                    current_words
                    > self.config.max_chunk_words
                ):
                    break

                boundary = boundaries.get(
                    end_index
                )

                if boundary is None:
                    continue

                required_min = (
                    self.config.strong_transition_min_words
                    if (
                        boundary.transition_strength
                        == "strong"
                    )
                    else self.config.min_chunk_words
                )

                if current_words < required_min:
                    continue

                words_after = words_between(
                    end_index + 1,
                    number_of_units - 1,
                )

                if (
                    0
                    < words_after
                    < self.config.min_chunk_words
                ):
                    continue

                candidates.append(
                    (
                        end_index,
                        boundary,
                    )
                )

            if candidates:
                def candidate_score(
                    candidate: tuple[
                        int,
                        _Boundary,
                    ],
                ) -> float:
                    end_index, boundary = candidate

                    current_words = words_between(
                        start,
                        end_index,
                    )

                    size_distance = abs(
                        current_words
                        - self.config.target_chunk_words
                    )

                    size_penalty = (
                        size_distance
                        / self.config.target_chunk_words
                    )

                    score = (
                        boundary.similarity
                        + (
                            self.config.size_penalty_weight
                            * size_penalty
                        )
                    )

                    if (
                        boundary.transition_strength
                        == "strong"
                    ):
                        score -= (
                            self.config.strong_transition_bonus
                        )
                    elif (
                        boundary.transition_strength
                        == "soft"
                    ):
                        score -= (
                            self.config.soft_transition_bonus
                        )

                    return score

                end, selected = min(
                    candidates,
                    key=candidate_score,
                )

                plans.append(
                    _ChunkPlan(
                        start_unit=start,
                        end_unit=end,
                        reason=selected.reason,
                        similarity=selected.similarity,
                        transition_strength=(
                            selected.transition_strength
                        ),
                    )
                )

                start = end + 1
                continue

            if (
                remaining_words
                <= self.config.max_chunk_words
            ):
                plans.append(
                    _ChunkPlan(
                        start_unit=start,
                        end_unit=(
                            number_of_units - 1
                        ),
                        reason="end_of_transcript",
                        similarity=None,
                        transition_strength=None,
                    )
                )
                break

            possible_ends: list[int] = []

            for end_index in range(
                start,
                number_of_units - 1,
            ):
                current_words = words_between(
                    start,
                    end_index,
                )

                if (
                    current_words
                    > self.config.max_chunk_words
                ):
                    break

                words_after = words_between(
                    end_index + 1,
                    number_of_units - 1,
                )

                if (
                    words_after
                    >= self.config.min_chunk_words
                ):
                    possible_ends.append(
                        end_index
                    )

            if possible_ends:
                end = possible_ends[-1]
            else:
                end = start

            if end < len(similarities):
                similarity: float | None = float(
                    similarities[end]
                )
            else:
                similarity = None

            plans.append(
                _ChunkPlan(
                    start_unit=start,
                    end_unit=end,
                    reason="max_size",
                    similarity=similarity,
                    transition_strength=None,
                )
            )

            start = end + 1

        return plans

    # -----------------------------------------------------------------
    # Materialize chunks + logical segment metadata
    # -----------------------------------------------------------------

    def _materialize_chunks(
        self,
        plans: list[_ChunkPlan],
        units: list[_SemanticUnit],
        sentences: list[str],
    ) -> list[TranscriptChunk]:
        chunks: list[TranscriptChunk] = []

        for index, plan in enumerate(
            plans
        ):
            core_start_sentence = (
                units[
                    plan.start_unit
                ].start_sentence
            )

            core_end_sentence = (
                units[
                    plan.end_unit
                ].end_sentence
            )

            text_start_sentence = (
                core_start_sentence
            )

            overlap_word_count = 0

            if (
                index > 0
                and plans[
                    index - 1
                ].reason == "max_size"
            ):
                (
                    text_start_sentence,
                    overlap_word_count,
                ) = self._find_overlap_start(
                    sentences=sentences,
                    core_start_sentence=(
                        core_start_sentence
                    ),
                )

            selected_sentences = sentences[
                text_start_sentence
                : core_end_sentence + 1
            ]

            text = " ".join(
                selected_sentences
            ).strip()

            chunks.append(
                TranscriptChunk(
                    chunk_id=index + 1,
                    text=text,
                    word_count=self._word_count(
                        text
                    ),
                    sentence_count=(
                        core_end_sentence
                        - text_start_sentence
                        + 1
                    ),
                    start_sentence=(
                        text_start_sentence
                    ),
                    end_sentence=(
                        core_end_sentence
                    ),
                    core_start_sentence=(
                        core_start_sentence
                    ),
                    core_end_sentence=(
                        core_end_sentence
                    ),
                    boundary_reason=(
                        plan.reason
                    ),
                    boundary_similarity=(
                        round(
                            plan.similarity,
                            4,
                        )
                        if (
                            plan.similarity
                            is not None
                        )
                        else None
                    ),
                    boundary_transition_strength=(
                        plan.transition_strength
                    ),
                    overlap_word_count=(
                        overlap_word_count
                    ),
                )
            )

        return self._assign_segment_metadata(
            chunks=chunks,
            plans=plans,
        )

    def _assign_segment_metadata(
        self,
        *,
        chunks: list[TranscriptChunk],
        plans: list[_ChunkPlan],
    ) -> list[TranscriptChunk]:
        """
        Group physical chunks into deterministic logical segments.

        Safe rule:
        - A chunk continues the same segment only when the previous
          chunk ended because of `max_size`.
        - Semantic and transition boundaries start a new segment.

        This avoids guessing topic identity inside Module 2 while still
        telling Module 3 which chunks are definitely continuations.
        """

        if not chunks:
            return []

        assignments: list[dict[str, object]] = []

        segment_number = 1
        segment_root_chunk_id = chunks[0].chunk_id
        segment_chunk_index = 1

        assignments.append(
            {
                "segment_number": segment_number,
                "segment_root_chunk_id": (
                    segment_root_chunk_id
                ),
                "segment_chunk_index": (
                    segment_chunk_index
                ),
                "is_continuation": False,
                "continuation_of_chunk_id": None,
                "continuation_reason": None,
            }
        )

        for index in range(
            1,
            len(chunks),
        ):
            previous_plan = plans[
                index - 1
            ]

            is_continuation = (
                previous_plan.reason
                == "max_size"
            )

            if is_continuation:
                segment_chunk_index += 1
            else:
                segment_number += 1
                segment_root_chunk_id = (
                    chunks[index].chunk_id
                )
                segment_chunk_index = 1

            assignments.append(
                {
                    "segment_number": segment_number,
                    "segment_root_chunk_id": (
                        segment_root_chunk_id
                    ),
                    "segment_chunk_index": (
                        segment_chunk_index
                    ),
                    "is_continuation": (
                        is_continuation
                    ),
                    "continuation_of_chunk_id": (
                        chunks[index - 1].chunk_id
                        if is_continuation
                        else None
                    ),
                    "continuation_reason": (
                        "max_size_split"
                        if is_continuation
                        else None
                    ),
                }
            )

        counts: dict[int, int] = defaultdict(int)

        for assignment in assignments:
            counts[
                int(
                    assignment[
                        "segment_number"
                    ]
                )
            ] += 1

        updated_chunks: list[
            TranscriptChunk
        ] = []

        for chunk, assignment in zip(
            chunks,
            assignments,
            strict=True,
        ):
            segment_number = int(
                assignment[
                    "segment_number"
                ]
            )

            segment_chunk_count = counts[
                segment_number
            ]

            segment_chunk_index = int(
                assignment[
                    "segment_chunk_index"
                ]
            )

            if segment_chunk_count == 1:
                segment_position = "single"
            elif segment_chunk_index == 1:
                segment_position = "start"
            elif (
                segment_chunk_index
                == segment_chunk_count
            ):
                segment_position = "end"
            else:
                segment_position = "middle"

            updates = {
                "segment_id": (
                    f"segment_{segment_number:03d}"
                ),
                "segment_root_chunk_id": int(
                    assignment[
                        "segment_root_chunk_id"
                    ]
                ),
                "segment_chunk_index": (
                    segment_chunk_index
                ),
                "segment_chunk_count": (
                    segment_chunk_count
                ),
                "segment_position": (
                    segment_position
                ),
                "is_continuation": bool(
                    assignment[
                        "is_continuation"
                    ]
                ),
                "continuation_of_chunk_id": (
                    assignment[
                        "continuation_of_chunk_id"
                    ]
                ),
                "continuation_reason": (
                    assignment[
                        "continuation_reason"
                    ]
                ),
            }

            if hasattr(
                chunk,
                "model_copy",
            ):
                updated = chunk.model_copy(
                    update=updates
                )
            else:
                updated = chunk.copy(
                    update=updates
                )

            updated_chunks.append(
                updated
            )

        return updated_chunks

    def _find_overlap_start(
        self,
        sentences: list[str],
        core_start_sentence: int,
    ) -> tuple[int, int]:
        if (
            self.config.max_size_overlap_words
            <= 0
            or self.config.max_size_overlap_sentences
            <= 0
            or core_start_sentence <= 0
        ):
            return (
                core_start_sentence,
                0,
            )

        overlap_start = (
            core_start_sentence
        )

        overlap_words = 0
        overlap_sentences = 0

        sentence_index = (
            core_start_sentence - 1
        )

        while (
            sentence_index >= 0
            and overlap_sentences
            < self.config.max_size_overlap_sentences
        ):
            sentence_words = self._word_count(
                sentences[
                    sentence_index
                ]
            )

            if (
                overlap_sentences > 0
                and (
                    overlap_words
                    + sentence_words
                )
                > self.config.max_size_overlap_words
            ):
                break

            overlap_start = sentence_index
            overlap_words += sentence_words
            overlap_sentences += 1
            sentence_index -= 1

            if (
                overlap_words
                >= self.config.max_size_overlap_words
            ):
                break

        return (
            overlap_start,
            overlap_words,
        )

    # -----------------------------------------------------------------
    # Utility
    # -----------------------------------------------------------------

    @staticmethod
    def _word_count(
        text: str,
    ) -> int:
        return len(
            re.findall(
                r"\S+",
                text,
            )
        )

# 4. Exact Existing Semantic Chunking Test Script

The project contains the command-line test:

```text
scripts/test_semantic_chunking.py
```

The exact source is shown below for documentation.

It is intentionally displayed as a **Markdown code block rather than an executable notebook cell** because its `main()` function uses `argparse` and expects a terminal input argument.

Executing the complete command-line script directly inside Jupyter would cause:

```text
SystemExit: 2
```

because the notebook kernel does not supply the required positional transcript path.

This does not change the script or the semantic chunking logic. The interactive cells after this section run the same existing `SemanticChunker` classes without invoking the command-line interface.

## Exact source: `scripts/test_semantic_chunking.py`

```python
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

from app.services.semantic_chunker import (
    SemanticChunker,
    SemanticChunkingConfig,
)


def save_json(
    result,
    output_path: Path,
) -> None:
    """Save the Pydantic chunking result as readable JSON."""

    if hasattr(
        result,
        "model_dump",
    ):
        data = result.model_dump()
    else:
        data = result.dict()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Test Module 2 semantic chunking and "
            "logical continuation metadata."
        )
    )

    parser.add_argument(
        "input",
        type=Path,
        help="Path to a cleaned transcript text file.",
    )

    parser.add_argument(
        "--output",
        type=Path,
        default=None,
    )

    parser.add_argument(
        "--repeat",
        type=int,
        default=1,
    )

    args = parser.parse_args()

    if args.repeat < 1:
        raise ValueError(
            "--repeat must be at least 1."
        )

    if not args.input.exists():
        raise FileNotFoundError(
            f"Input file not found: {args.input}"
        )

    cleaned_text = args.input.read_text(
        encoding="utf-8"
    ).strip()

    if not cleaned_text:
        raise ValueError(
            "Cleaned transcript file is empty."
        )

    config = SemanticChunkingConfig(
        min_chunk_words=150,
        target_chunk_words=325,
        max_chunk_words=550,
        strong_transition_min_words=80,
        semantic_unit_words=60,
        boundary_percentile=15.0,
        threshold_floor=0.10,
        threshold_ceiling=0.45,
        soft_transition_margin=0.10,
        soft_transition_similarity_ceiling=0.35,
        max_size_overlap_words=45,
        max_size_overlap_sentences=2,
    )

    chunker = SemanticChunker(
        config=config
    )

    result = None
    timings: list[float] = []

    for run_number in range(
        1,
        args.repeat + 1,
    ):
        start = time.perf_counter()

        result = chunker.chunk(
            cleaned_text
        )

        timings.append(
            time.perf_counter()
            - start
        )

        print(
            f"Run {run_number}: "
            f"{timings[-1]:.3f} seconds"
        )

    if result is None:
        raise RuntimeError(
            "Chunking did not produce a result."
        )

    print()
    print("=" * 100)
    print(
        "MODULE 2 — CHUNKS + SEGMENT METADATA"
    )
    print("=" * 100)
    print(
        f"Embedding model: {result.embedding_model}"
    )
    print(
        f"Semantic threshold: {result.semantic_threshold}"
    )
    print(
        f"Physical chunks: {len(result.chunks)}"
    )
    print(
        f"Logical segments: {result.segment_count}"
    )

    for chunk in result.chunks:
        print()
        print("-" * 100)
        print(
            f"CHUNK {chunk.chunk_id}"
        )
        print(
            f"Segment: {chunk.segment_id}"
        )
        print(
            "Segment position: "
            f"{chunk.segment_position} "
            f"({chunk.segment_chunk_index}/"
            f"{chunk.segment_chunk_count})"
        )
        print(
            "Segment root chunk: "
            f"{chunk.segment_root_chunk_id}"
        )
        print(
            "Is continuation: "
            f"{chunk.is_continuation}"
        )
        print(
            "Continuation of chunk: "
            f"{chunk.continuation_of_chunk_id}"
        )
        print(
            "Continuation reason: "
            f"{chunk.continuation_reason}"
        )
        print(
            f"Boundary reason: {chunk.boundary_reason}"
        )
        print(
            f"Words: {chunk.word_count}"
        )
        print("-" * 100)
        print(chunk.text)

    output_path = (
        args.output
        if args.output
        else args.input.with_name(
            args.input.stem
            + "_chunks.json"
        )
    )

    save_json(
        result,
        output_path,
    )

    print()
    print(
        f"Chunk JSON saved to: {output_path}"
    )


if __name__ == "__main__":
    main()
```

### Terminal usage

The original script can still be executed from the project terminal:

```powershell
python -m scripts.test_semantic_chunking "test_data/transcript_1_cleaned.txt"
```

The notebook execution below uses the same configuration and the same project classes interactively.

# 5. Use the Existing Project Classes

The command-line wrapper is not executed inside Jupyter.

The notebook now imports the same existing project classes and runs them interactively:

- `ChunkingResult`
- `SemanticChunker`
- `SemanticChunkingConfig`

This preserves the implementation while replacing only the terminal-specific `argparse` entry point with notebook cells.

In [ ]:
import json
import time
from collections import Counter
from typing import Any

import numpy as np
import pandas as pd

from app.schemas.chunk import ChunkingResult
from app.services.embedding_service import (
    CHUNKING_EMBEDDING_MODEL,
    DEFAULT_EMBEDDING_MODEL,
)
from app.services.semantic_chunker import (
    SemanticChunker,
    SemanticChunkingConfig,
)

print("Project Module 2 classes imported successfully.")
print(f"Qwen model: {DEFAULT_EMBEDDING_MODEL}")
print(f"MiniLM model: {CHUNKING_EMBEDDING_MODEL}")

# 6. Existing Chunking Configuration

The following values are the same values already used by the existing Agent 1 semantic chunking test and pipeline.

Both embedding-model runs use these exact values.

Only `embedding_model` changes.

In [ ]:
COMMON_CONFIG = {
    "min_chunk_words": 150,
    "target_chunk_words": 325,
    "max_chunk_words": 550,
    "strong_transition_min_words": 80,
    "semantic_unit_words": 60,
    "boundary_percentile": 15.0,
    "threshold_floor": 0.10,
    "threshold_ceiling": 0.45,
    "soft_transition_margin": 0.10,
    "soft_transition_similarity_ceiling": 0.35,
    "max_size_overlap_words": 45,
    "max_size_overlap_sentences": 2,
}

minilm_config = SemanticChunkingConfig(
    **COMMON_CONFIG,
    embedding_model=CHUNKING_EMBEDDING_MODEL,
)

qwen_config = SemanticChunkingConfig(
    **COMMON_CONFIG,
    embedding_model=DEFAULT_EMBEDDING_MODEL,
)

minilm_chunker = SemanticChunker(
    config=minilm_config
)

qwen_chunker = SemanticChunker(
    config=qwen_config
)

print("MiniLM configuration:")
print(minilm_config)

print("\nQwen configuration:")
print(qwen_config)

# 7. Select a Real Module 1 Cleaned Transcript

This cell searches only for real cleaned outputs already produced by Module 1.

No transcript is generated inside this notebook.

In [ ]:
cleaned_candidates = [
    *(
        PROJECT_ROOT
        / "test_outputs"
        / "pipeline_runs"
    ).glob(
        "*/01_preprocessing/cleaned_transcript.txt"
    ),
    *(
        PROJECT_ROOT
        / "test_outputs"
        / "module_1_2_batch"
    ).glob(
        "*/cleaned.txt"
    ),
]

fallback_path = (
    PROJECT_ROOT
    / "test_data"
    / "transcript_1_cleaned.txt"
)

if fallback_path.exists():
    cleaned_candidates.append(
        fallback_path
    )

cleaned_paths = sorted(
    {
        path.resolve()
        for path in cleaned_candidates
        if (
            path.exists()
            and path.is_file()
            and path.read_text(
                encoding="utf-8"
            ).strip()
        )
    },
    key=lambda path: str(path).casefold(),
)

if not cleaned_paths:
    raise FileNotFoundError(
        "No real cleaned transcript was found."
    )

for index, path in enumerate(
    cleaned_paths,
    start=1,
):
    print(
        f"{index:02d}. "
        f"{path.relative_to(PROJECT_ROOT)}"
    )

PRIMARY_TRANSCRIPT_PATH = cleaned_paths[0]

cleaned_text = (
    PRIMARY_TRANSCRIPT_PATH
    .read_text(encoding="utf-8")
    .strip()
)

print(
    "\nSelected transcript:",
    PRIMARY_TRANSCRIPT_PATH.relative_to(
        PROJECT_ROOT
    ),
)

print(
    "Words:",
    len(cleaned_text.split()),
)

# 8. Run MiniLM with the Existing Chunker

This runs the exact current `SemanticChunker` using the final MiniLM model.

In [ ]:
minilm_start = time.perf_counter()

minilm_result = minilm_chunker.chunk(
    cleaned_text
)

minilm_runtime = (
    time.perf_counter()
    - minilm_start
)

print(
    json.dumps(
        minilm_result.model_dump(),
        indent=2,
        ensure_ascii=False,
    )
)

print(
    f"\nMiniLM runtime: "
    f"{minilm_runtime:.4f} seconds"
)

# 9. Run Qwen3-Embedding-0.6B with the Same Existing Chunker

This cell changes only the existing `embedding_model` configuration.

All semantic chunking logic and all other configuration values remain identical.

In [ ]:
qwen_start = time.perf_counter()

qwen_result = qwen_chunker.chunk(
    cleaned_text
)

qwen_runtime = (
    time.perf_counter()
    - qwen_start
)

print(
    json.dumps(
        qwen_result.model_dump(),
        indent=2,
        ensure_ascii=False,
    )
)

print(
    f"\nQwen runtime: "
    f"{qwen_runtime:.4f} seconds"
)

# 10. Compare the Real Outputs

This section does not influence chunk generation.

It only reads the two real `ChunkingResult` objects and compares:

- execution time;
- physical chunk count;
- logical segment count;
- semantic threshold;
- boundary reasons;
- continuation chunks;
- overlap chunks;
- chunk sizes;
- boundary-position agreement.

No expected result is hardcoded.

In [ ]:
def real_result_metrics(
    result: ChunkingResult,
) -> dict[str, Any]:
    chunk_word_counts = [
        chunk.word_count
        for chunk in result.chunks
    ]

    return {
        "embedding_model": (
            result.embedding_model
        ),
        "total_words": (
            result.total_words
        ),
        "total_sentences": (
            result.total_sentences
        ),
        "semantic_units": (
            result.semantic_unit_count
        ),
        "semantic_threshold": (
            result.semantic_threshold
        ),
        "physical_chunks": len(
            result.chunks
        ),
        "logical_segments": (
            result.segment_count
        ),
        "boundary_reasons": dict(
            Counter(
                chunk.boundary_reason
                for chunk in result.chunks
            )
        ),
        "continuation_chunks": sum(
            chunk.is_continuation
            for chunk in result.chunks
        ),
        "overlap_chunks": sum(
            chunk.overlap_word_count > 0
            for chunk in result.chunks
        ),
        "minimum_chunk_words": min(
            chunk_word_counts
        ),
        "maximum_chunk_words": max(
            chunk_word_counts
        ),
        "average_chunk_words": round(
            float(
                np.mean(
                    chunk_word_counts
                )
            ),
            2,
        ),
    }


def real_boundary_positions(
    result: ChunkingResult,
) -> set[int]:
    return {
        chunk.core_end_sentence
        for chunk in result.chunks
        if (
            chunk.boundary_reason
            != "end_of_transcript"
        )
    }


minilm_boundaries = (
    real_boundary_positions(
        minilm_result
    )
)

qwen_boundaries = (
    real_boundary_positions(
        qwen_result
    )
)

all_boundaries = (
    minilm_boundaries
    | qwen_boundaries
)

shared_boundaries = (
    minilm_boundaries
    & qwen_boundaries
)

boundary_agreement = (
    len(shared_boundaries)
    / len(all_boundaries)
    if all_boundaries
    else 1.0
)

comparison = {
    "source_transcript": str(
        PRIMARY_TRANSCRIPT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "same_chunking_configuration": (
        COMMON_CONFIG
    ),
    "minilm": {
        **real_result_metrics(
            minilm_result
        ),
        "runtime_seconds": round(
            minilm_runtime,
            4,
        ),
    },
    "qwen": {
        **real_result_metrics(
            qwen_result
        ),
        "runtime_seconds": round(
            qwen_runtime,
            4,
        ),
    },
    "comparison": {
        "qwen_to_minilm_runtime_ratio": (
            round(
                qwen_runtime
                / minilm_runtime,
                2,
            )
            if minilm_runtime > 0
            else None
        ),
        "boundary_position_agreement": round(
            boundary_agreement,
            4,
        ),
        "same_physical_chunk_count": (
            len(minilm_result.chunks)
            == len(qwen_result.chunks)
        ),
        "same_logical_segment_count": (
            minilm_result.segment_count
            == qwen_result.segment_count
        ),
        "same_total_words": (
            minilm_result.total_words
            == qwen_result.total_words
        ),
    },
}

print(
    json.dumps(
        comparison,
        indent=2,
        ensure_ascii=False,
    )
)

## Comparison table

The table below is derived directly from the two actual results.

In [ ]:
comparison_table = pd.DataFrame(
    [
        {
            "Model": "MiniLM",
            "Model ID": (
                CHUNKING_EMBEDDING_MODEL
            ),
            "Runtime seconds": round(
                minilm_runtime,
                4,
            ),
            **real_result_metrics(
                minilm_result
            ),
        },
        {
            "Model": "Qwen3-Embedding-0.6B",
            "Model ID": (
                DEFAULT_EMBEDDING_MODEL
            ),
            "Runtime seconds": round(
                qwen_runtime,
                4,
            ),
            **real_result_metrics(
                qwen_result
            ),
        },
    ]
)

comparison_table

# 11. Run Both Models on Every Real Cleaned Transcript

This batch uses the same existing chunker and the same real cleaned transcript files.

A failure on one model or one transcript does not stop the remaining runs.

In [ ]:
RUN_QWEN_FULL_BATCH = True

batch_models = [
    (
        "MiniLM",
        minilm_chunker,
    ),
]

if RUN_QWEN_FULL_BATCH:
    batch_models.append(
        (
            "Qwen3-Embedding-0.6B",
            qwen_chunker,
        )
    )

batch_results = []

for transcript_path in cleaned_paths:
    transcript_text = (
        transcript_path
        .read_text(encoding="utf-8")
        .strip()
    )

    for model_label, chunker in batch_models:
        row = {
            "transcript": str(
                transcript_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "model": model_label,
            "status": "running",
        }

        try:
            start = time.perf_counter()

            result = chunker.chunk(
                transcript_text
            )

            runtime = (
                time.perf_counter()
                - start
            )

            row.update(
                {
                    "status": "completed",
                    "runtime_seconds": round(
                        runtime,
                        4,
                    ),
                    **real_result_metrics(
                        result
                    ),
                }
            )

            print(
                f"{model_label} | "
                f"{transcript_path.name} | "
                f"{runtime:.4f}s | "
                f"{len(result.chunks)} chunks"
            )

        except Exception as exc:
            row.update(
                {
                    "status": "failed",
                    "error_type": (
                        type(exc).__name__
                    ),
                    "error_message": str(exc),
                }
            )

            print(
                f"{model_label} | "
                f"{transcript_path.name} | "
                f"FAILED: {exc}"
            )

        batch_results.append(row)

# 12. Save and Summarise the Real Batch

The saved files contain only measured outputs from the existing implementation.

In [ ]:
OUTPUT_ROOT = (
    PROJECT_ROOT
    / "test_outputs"
    / "module_2_exact_notebook"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

batch_json_path = (
    OUTPUT_ROOT
    / "batch_results.json"
)

batch_json_path.write_text(
    json.dumps(
        batch_results,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

batch_dataframe = pd.DataFrame(
    batch_results
)

batch_csv_path = (
    OUTPUT_ROOT
    / "batch_results.csv"
)

batch_dataframe.to_csv(
    batch_csv_path,
    index=False,
)

print(f"JSON: {batch_json_path}")
print(f"CSV: {batch_csv_path}")

batch_dataframe

# 13. Final Decision

## Existing logic

The notebook uses the current project logic without rewriting:

- chunk schemas;
- embedding service;
- semantic chunker;
- chunking configuration;
- sentence splitting;
- semantic units;
- similarity calculation;
- threshold calculation;
- transition rules;
- boundary planning;
- overlap;
- logical segment metadata.

## Model comparison

Qwen3-Embedding-0.6B and MiniLM perform the same embedding role.

The comparison changes only:

```python
embedding_model
```

The final model decision should be based on the measured real outputs:

- runtime;
- chunk structure;
- logical segment structure;
- threshold behaviour;
- boundary agreement;
- practical processing cost.

MiniLM is selected when it produces acceptable real chunking results with significantly lower execution time than Qwen.